<a href="https://colab.research.google.com/github/ArkanDash/Advanced-RVC-Inference/blob/master/colab-noui.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


<h1><font color="#a78bfa">🎙️ Advanced RVC Inference v2.1.0</font></h1>
<h3><font color="#c4b5fd">CLI-Only Colab — Full Parameter Control</font></h3>
<p>Voice Conversion · Audio Separation · TTS · Training · Model Management</p>
<hr>
<p>
  <a href="https://discord.gg/hvmsukmBHE"><font color="#a78bfa">Discord</font></a> ·
  <a href="https://github.com/ArkanDash/Advanced-RVC-Inference"><font color="#a78bfa">Github</font></a> ·
  <a href="https://github.com/ArkanDash/Advanced-RVC-Inference/blob/master/docs/README.md"><font color="#a78bfa">CLI Guide</font></a> ·
  <a href="https://colab.research.google.com/github/ArkanDash/Advanced-RVC-Inference/blob/master/Advanced-RVC.ipynb"><font color="#a78bfa">Main Colab</font></a>
</p>


<h2><font color="#9ca3af">⚙️ Setup</font></h2>
<p>Mount Google Drive and clone the repo locally. This notebook skips the Gradio web UI for a lighter, faster setup.</p>


In [ ]:
# @title Mount Google Drive
from google.colab import drive
from google.colab._message import MessageError

try:
  drive.mount("/content/drive")
except MessageError:
  print("❌ Failed to mount drive")

In [ ]:
# @title Install Dependencies
from IPython.display import clear_output
import os

REPO_DIR = "/content/Advanced-RVC-Inference"
LOGS_PATH = f"{REPO_DIR}/arvc/assets/logs"
BACKUPS_PATH = "/content/drive/MyDrive/RVCBackup"
CLI = "python -m arvc.api.cli"

# Install system dependencies
!apt-get -y install libportaudio2 ffmpeg git -qq > /dev/null 2>&1

# Install uv for faster package management
!pip install uv -q

# Clone or update the repo
if os.path.exists(REPO_DIR):
    %cd $REPO_DIR
    !git pull origin master -q
else:
    !git clone https://github.com/ArkanDash/Advanced-RVC-Inference.git /content/Advanced-RVC-Inference -q
    %cd $REPO_DIR

# Install Python dependencies
!uv pip install -r requirements.txt --system -q 2>/dev/null || uv pip install -r requirements.txt --system -q

# Ensure the package itself is importable
%cd $REPO_DIR
import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

clear_output()
print("✅ Advanced RVC Inference installed!")
print("📖 Full CLI guide: https://github.com/ArkanDash/Advanced-RVC-Inference/blob/master/docs/README.md")
print()
print("Run `python -m arvc.api.cli --help` to see all available commands.")

In [ ]:
# @title Set Environment Variables
# @markdown Tune these for better performance or compatibility.
import os
import sys

# Ensure working directory is always the repo root
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['FAISS_DISABLE_CPU_FEATURES'] = 'AVX512,AVX2,AVX512_SPR,SVE'

print("✅ Environment variables set.")
print(f"  TF_CPP_MIN_LOG_LEVEL = {os.environ['TF_CPP_MIN_LOG_LEVEL']}")
print(f"  PYTORCH_ENABLE_MPS_FALLBACK = {os.environ['PYTORCH_ENABLE_MPS_FALLBACK']}")
print(f"  CUDA_LAUNCH_BLOCKING = {os.environ['CUDA_LAUNCH_BLOCKING']}")
print(f"  Working directory: {os.getcwd()}")

<h2><font color="#38ef7d">📋 System Info</font></h2>
<p>Check your GPU, installed models, and available F0 methods.</p>


In [ ]:
# @title System Info
import os, sys
os.chdir(REPO_DIR)
!python -m arvc.api.cli info

In [ ]:
# @title List Installed Models
import os
os.chdir(REPO_DIR)
!python -m arvc.api.cli list-models

In [ ]:
# @title List F0 Methods
import os
os.chdir(REPO_DIR)
!python -m arvc.api.cli list-f0-methods

In [ ]:
# @title Show Version
import os
os.chdir(REPO_DIR)
!python -m arvc.api.cli version

<h2><font color="#6dd5ed">⬇️ Download Models & Audio</font></h2>
<p>Download RVC models from HuggingFace or other supported sources, search HuggingFace for models, download audio from YouTube, and download pretrained base models.</p>


In [ ]:
# @title Download Model from URL
# @markdown Paste a HuggingFace, Google Drive, MediaFire, or Mega URL.
download_url = ""  # @param {type:"string"}
model_name = ""  # @param {type:"string"}

import os
os.chdir(REPO_DIR)

cmd = f"python -m arvc.api.cli download -l \"{download_url}\""
if model_name:
    cmd += f" -n \"{model_name}\""

!{cmd}

In [ ]:
# @title Search & Download Models from HuggingFace
# @markdown Search for RVC models on HuggingFace and download them.
search_query = ""  # @param {type:"string"}

import os, json, requests
os.chdir(REPO_DIR)

# Search HuggingFace API
api_url = f"https://huggingface.co/api/models?search=rvc+{search_query}&sort=downloads&limit=10"
try:
    resp = requests.get(api_url, timeout=15)
    results = resp.json()
    if not results:
        print("No models found.")
    else:
        print(f"Found {len(results)} models:\n")
        for i, m in enumerate(results):
            model_id = m.get('modelId', 'unknown')
            downloads = m.get('downloads', 0)
            print(f"  {i+1}. {model_id}  ({downloads:,} downloads)")
        print()
        print("To download, copy the model ID and use the 'Download Model from URL' cell above.")
        print("Example URL format: https://huggingface.co/MODEL_ID/resolve/main/MODEL_NAME.pth")
except Exception as e:
    print(f"Search failed: {e}")

In [ ]:
# @title Download Audio from URL
# @markdown Download audio from YouTube or other supported sites.
audio_url = ""  # @param {type:"string"}
output_dir = "/content/Advanced-RVC-Inference/arvc/assets/audios"  # @param {type:"string"}

import os
os.chdir(REPO_DIR)

if not audio_url:
    print("❌ Enter a URL.")
else:
    !yt-dlp -x --audio-format wav -o "{output_dir}/%(title)s.%(ext)s" "{audio_url}"
    print("\n✅ Audio downloaded.")

In [ ]:
# @markdown ---
# @markdown ### 📥 Pretrained Model Selection
# @title Download Pretrained Models
# @markdown Download pretrained RVC base models needed for training.
pretrained_model = "v2"  # @param ["v1", "v2"]
pretrained_sr = "48k"  # @param ["32k", "40k", "48k"]

import os
os.chdir(REPO_DIR)

from arvc.services.downloads import download_pretrained_model
from arvc.utils.variables import translations

result = download_pretrained_model(
    choices=translations["list_model"],
    model=pretrained_model,
    sample_rate=pretrained_sr
)
print(result if result else "✅ Pretrained models downloaded.")

<h2><font color="#764ba2">🎤 Voice Conversion (Inference)</font></h2>
<p>Convert voice in audio files using an RVC model — with <b>full parameter control</b> matching the Gradio UI.</p>


In [ ]:
# @markdown ---
# @markdown ### 🎯 Core Settings
# @title Voice Conversion (Full Parameters)
# @markdown Convert audio using an RVC model with all parameters from the Gradio UI.

# --- Core ---
model_path = ""  # @param {type:"string"}
index_path = ""  # @param {type:"string"}
input_audio = ""  # @param {type:"string"}
output_audio = ""  # @param {type:"string"}
pitch_shift = 0  # @param {type:"slider", min:-24, max:24, step:1}
f0_method = "rmvpe"  # @param ["rmvpe", "crepe-full", "crepe-tiny", "fcpe", "harvest", "pyin", "mangio-crepe-full", "hybrid"]
output_format = "wav"  # @param ["wav", "mp3", "flac", "ogg"]

# @markdown ---
# @markdown ### 🔧 F0 & Index Settings
# --- F0 Settings ---
filter_radius = 3  # @param {type:"slider", min:0, max:10, step:1}
index_rate = 0.5  # @param {type:"slider", min:0, max:1, step:0.01}
rms_mix_rate = 1.0  # @param {type:"slider", min:0, max:1, step:0.01}
protect = 0.33  # @param {type:"slider", min:0, max:0.5, step:0.01}
hop_length = 64  # @param {type:"slider", min:32, max:512, step:32}
f0_autotune = False  # @param {type:"boolean"}
f0_autotune_strength = 1.0  # @param {type:"slider", min:0, max:1, step:0.01}
predictor_onnx = False  # @param {type:"boolean"}

# @markdown ---
# @markdown ### 🔀 Hybrid F0 Settings
# --- Hybrid F0 (only when f0_method=hybrid) ---
hybrid_method = "hybrid[rmvpe+crepe-full]"  # @param {type:"string"}
alpha = 0.5  # @param {type:"slider", min:0, max:1, step:0.01}

# @markdown ---
# @markdown ### 🧠 Embedder Settings
# --- Embedder ---
embedder_model = "hubert_base"  # @param ["hubert_base", "contentvec_base", "custom"]
embedders_mode = "fairseq"  # @param ["fairseq", "transformers", "onnx", "whisper"]
custom_embedder_name = ""  # @param {type:"string"}

# @markdown ---
# @markdown ### 🎚️ Audio Processing
# --- Audio Processing ---
split_audio = False  # @param {type:"boolean"}
formant_shifting = False  # @param {type:"boolean"}
formant_qfrency = 0.8  # @param {type:"slider", min:0, max:2, step:0.01}
formant_timbre = 0.8  # @param {type:"slider", min:0, max:2, step:0.01}
clean_audio = False  # @param {type:"boolean"}
clean_strength = 0.7  # @param {type:"slider", min:0, max:1, step:0.01}
checkpointing = False  # @param {type:"boolean"}
resample_sr = 0  # @param [0, 32000, 40000, 44100, 48000]

# @markdown ---
# @markdown ### ⚡ Advanced Settings
# --- Advanced ---
f0_file = ""  # @param {type:"string"}
proposal_pitch = False  # @param {type:"boolean"}
proposal_pitch_threshold = 255.0  # @param {type:"slider", min:50, max:1200, step:0.1}
audio_processing = False  # @param {type:"boolean"}
speaker_id = 0  # @param {type:"slider", min:0, max:10, step:1}

import os
os.chdir(REPO_DIR)

# Use direct Python API for full parameter support
from arvc.engine.inference.inference import convert

# Set default output path
if not output_audio:
    from pathlib import Path
    stem = Path(input_audio).stem
    output_audio = str(Path(input_audio).parent / f"{stem}_converted.{output_format}")

# Resolve embedder
embedder = custom_embedder_name if embedder_model == "custom" and custom_embedder_name else embedder_model

convert(
    pitch=pitch_shift,
    filter_radius=filter_radius,
    index_rate=index_rate,
    rms_mix_rate=rms_mix_rate,
    protect=protect,
    hop_length=hop_length,
    f0_method=f0_method,
    input_path=input_audio,
    output_path=output_audio,
    pth_path=model_path,
    index_path=index_path,
    f0_autotune=f0_autotune,
    clean_audio=clean_audio,
    clean_strength=clean_strength,
    export_format=output_format,
    embedder_model=embedder,
    resample_sr=resample_sr,
    split_audio=split_audio,
    f0_autotune_strength=f0_autotune_strength,
    checkpointing=checkpointing,
    predictor_onnx=predictor_onnx,
    embedders_mode=embedders_mode,
    formant_shifting=formant_shifting,
    formant_qfrency=formant_qfrency,
    formant_timbre=formant_timbre,
    f0_file=f0_file or "",
    proposal_pitch=proposal_pitch,
    proposal_pitch_threshold=proposal_pitch_threshold,
    audio_processing=audio_processing,
    alpha=alpha,
    sid=speaker_id,
)

print(f"\n✅ Output saved to: {output_audio}")

<h2><font color="#f5576c">🔊 Text-to-Speech + RVC Conversion</font></h2>
<p>Generate speech from text using Edge TTS or Google TTS, then convert it with an RVC model.<br>Mirrors the <b>Convert Text</b> tab from the Gradio UI.</p>


In [ ]:
# @markdown ---
# @markdown ### 🗣️ TTS Settings
# @title Text-to-Speech (Edge TTS / Google TTS)
# @markdown Generate speech audio from text using Edge TTS or Google TTS.
text = "Hello, this is a test of text to speech conversion."  # @param {type:"raw"}
tts_voice = "en-US-AriaNeural"  # @param {type:"string"}
use_google_tts = False  # @param {type:"boolean"}
speed = 0  # @param {type:"slider", min:-10, max:10, step:1}
tts_pitch = 0  # @param {type:"slider", min:-10, max:10, step:1}
tts_output = "arvc/assets/audios/tts/tts.wav"  # @param {type:"string"}

import os
os.chdir(REPO_DIR)

from arvc.services.tts import TTS

result = TTS(
    prompt=text,
    voice=tts_voice,
    speed=speed,
    output=tts_output,
    pitch=tts_pitch,
    google=use_google_tts,
    srt_input="",
)

print(f"\n✅ TTS audio saved to: {tts_output}")

In [ ]:
# @markdown ---
# @markdown ### 📂 Model & Input
# @title TTS + RVC Convert
# @markdown Convert TTS audio with an RVC model (same as TTS tab → Convert in the UI).

# --- Model ---
model_path = ""  # @param {type:"string"}
index_path = ""  # @param {type:"string"}

# --- TTS Input ---
tts_input = "arvc/assets/audios/tts/tts.wav"  # @param {type:"string"}
tts_convert_output = "arvc/assets/audios/rvc/tts-convert.wav"  # @param {type:"string"}

# @markdown ---
# @markdown ### 🎛️ Conversion Parameters
# --- Conversion Params ---
pitch_shift = 0  # @param {type:"slider", min:-24, max:24, step:1}
f0_method = "rmvpe"  # @param ["rmvpe", "crepe-full", "crepe-tiny", "fcpe", "harvest", "pyin", "hybrid"]
export_format = "wav"  # @param ["wav", "mp3", "flac", "ogg"]
filter_radius = 3  # @param {type:"slider", min:0, max:10, step:1}
index_rate = 0.5  # @param {type:"slider", min:0, max:1, step:0.01}
rms_mix_rate = 1.0  # @param {type:"slider", min:0, max:1, step:0.01}
protect = 0.5  # @param {type:"slider", min:0, max:0.5, step:0.01}
hop_length = 64  # @param {type:"slider", min:32, max:512, step:32}
split_audio = False  # @param {type:"boolean"}
f0_autotune = False  # @param {type:"boolean"}
f0_autotune_strength = 1.0  # @param {type:"slider", min:0, max:1, step:0.01}
formant_shifting = False  # @param {type:"boolean"}
formant_qfrency = 0.8  # @param {type:"slider", min:0, max:2, step:0.01}
formant_timbre = 0.8  # @param {type:"slider", min:0, max:2, step:0.01}
clean_audio = False  # @param {type:"boolean"}
clean_strength = 0.5  # @param {type:"slider", min:0, max:1, step:0.01}
checkpointing = False  # @param {type:"boolean"}
predictor_onnx = False  # @param {type:"boolean"}
embedder_model = "hubert_base"  # @param {type:"string"}
embedders_mode = "fairseq"  # @param ["fairseq", "transformers", "onnx", "whisper"]
resample_sr = 0  # @param [0, 32000, 40000, 44100, 48000]
f0_file = ""  # @param {type:"string"}
proposal_pitch = False  # @param {type:"boolean"}
proposal_pitch_threshold = 255.0  # @param {type:"slider", min:50, max:1200, step:0.1}
audio_processing = False  # @param {type:"boolean"}
alpha = 0.5  # @param {type:"slider", min:0, max:1, step:0.01}

import os
os.chdir(REPO_DIR)

from arvc.engine.inference.inference import convert_tts

result = convert_tts(
    clean_audio=clean_audio,
    autotune=f0_autotune,
    pitch=pitch_shift,
    clean_strength=clean_strength,
    model=model_path,
    index=index_path,
    index_rate=index_rate,
    input_path=tts_input,
    output_path=tts_convert_output,
    format=export_format,
    method=f0_method,
    hybrid_method="",
    hop_length=hop_length,
    embedders=embedder_model,
    custom_embedders="",
    resample_sr=resample_sr,
    filter_radius=filter_radius,
    rms_mix_rate=rms_mix_rate,
    protect=protect,
    split_audio=split_audio,
    f0_autotune_strength=f0_autotune_strength,
    checkpointing=checkpointing,
    predictor_onnx=predictor_onnx,
    formant_shifting=formant_shifting,
    formant_qfrency=formant_qfrency,
    formant_timbre=formant_timbre,
    f0_file=f0_file or "",
    embedders_mode=embedders_mode,
    proposal_pitch=proposal_pitch,
    proposal_pitch_threshold=proposal_pitch_threshold,
    audio_processing=audio_processing,
    alpha=alpha,
)

print(f"\n✅ TTS + RVC output saved to: {tts_convert_output}")

In [ ]:
# @title Google Translate Text
# @markdown Translate text before TTS (mirrors the Translate feature in the TTS tab).
text_to_translate = ""  # @param {type:"string"}
source_lang = "auto"  # @param {type:"string"}
target_lang = "en"  # @param {type:"string"}

import os
os.chdir(REPO_DIR)

from arvc.services.utils import google_translate

result = google_translate(text_to_translate, source_lang, target_lang)
print(f"Translated: {result}")

<h2><font color="#00f2fe">🎙️ Convert with Whisper (Multi-Speaker)</font></h2>
<p>Use Whisper for speaker diarization, then apply different RVC models to each speaker.<br>Mirrors the <b>Convert with Whisper</b> tab from the Gradio UI.</p>


In [ ]:
# @markdown ---
# @markdown ### 🎭 Speaker Models
# @title Convert with Whisper (Multi-Speaker Diarization)
# @markdown Separate speakers using Whisper diarization, then convert each with a different RVC model.

# --- Models (Speaker 1 & 2) ---
model_pth_1 = ""  # @param {type:"string"}
model_index_1 = ""  # @param {type:"string"}
model_pth_2 = ""  # @param {type:"string"}
model_index_2 = ""  # @param {type:"string"}

# @markdown ---
# @markdown ### 🎵 Audio Settings
# --- Audio ---
input_audio = ""  # @param {type:"string"}
output_audio = "arvc/assets/audios/rvc/whisper-output.wav"  # @param {type:"string"}
export_format = "wav"  # @param ["wav", "mp3", "flac", "ogg"]

# @markdown ---
# @markdown ### 🎤 Diarization Settings
# --- Diarization ---
num_speakers = 2  # @param {type:"slider", min:1, max:4, step:1}
whisper_model_size = "medium"  # @param ["tiny", "tiny.en", "base", "base.en", "small", "small.en", "medium", "medium.en", "large-v1", "large-v2", "large-v3", "large-v3-turbo"]

# @markdown ---
# @markdown ### 🎛️ Conversion Parameters
# --- Conversion Params ---
pitch_1 = 0  # @param {type:"slider", min:-24, max:24, step:1}
pitch_2 = 0  # @param {type:"slider", min:-24, max:24, step:1}
index_strength_1 = 0.5  # @param {type:"slider", min:0, max:1, step:0.01}
index_strength_2 = 0.5  # @param {type:"slider", min:0, max:1, step:0.01}
f0_method = "rmvpe"  # @param ["rmvpe", "crepe-full", "crepe-tiny", "fcpe", "harvest", "pyin", "hybrid"]
filter_radius = 3  # @param {type:"slider", min:0, max:10, step:1}
rms_mix_rate = 1.0  # @param {type:"slider", min:0, max:1, step:0.01}
protect = 0.5  # @param {type:"slider", min:0, max:0.5, step:0.01}
hop_length = 64  # @param {type:"slider", min:32, max:512, step:32}
embedder_model = "hubert_base"  # @param {type:"string"}
embedders_mode = "fairseq"  # @param ["fairseq", "transformers", "onnx", "whisper"]
resample_sr = 0  # @param [0, 32000, 40000, 44100, 48000]
f0_autotune = False  # @param {type:"boolean"}
f0_autotune_strength = 1.0  # @param {type:"slider", min:0, max:1, step:0.01}
clean_audio = False  # @param {type:"boolean"}
clean_strength = 0.5  # @param {type:"slider", min:0, max:1, step:0.01}
checkpointing = False  # @param {type:"boolean"}
predictor_onnx = False  # @param {type:"boolean"}
formant_shifting = False  # @param {type:"boolean"}
formant_qfrency_1 = 1.0  # @param {type:"number"}
formant_timbre_1 = 1.0  # @param {type:"number"}
formant_qfrency_2 = 1.0  # @param {type:"number"}
formant_timbre_2 = 1.0  # @param {type:"number"}
proposal_pitch = False  # @param {type:"boolean"}
proposal_pitch_threshold = 255.0  # @param {type:"slider", min:50, max:1200, step:0.1}
audio_processing = False  # @param {type:"boolean"}
alpha = 0.5  # @param {type:"slider", min:0, max:1, step:0.01}

import os
os.chdir(REPO_DIR)

from arvc.engine.inference.inference import convert_with_whisper

result = convert_with_whisper(
    num_spk=num_speakers,
    model_size=whisper_model_size,
    cleaner=clean_audio,
    clean_strength=clean_strength,
    autotune=f0_autotune,
    f0_autotune_strength=f0_autotune_strength,
    checkpointing=checkpointing,
    model_1=model_pth_1,
    model_2=model_pth_2,
    model_index_1=model_index_1,
    model_index_2=model_index_2,
    pitch_1=pitch_1,
    pitch_2=pitch_2,
    index_strength_1=index_strength_1,
    index_strength_2=index_strength_2,
    export_format=export_format,
    input_audio=input_audio,
    output_audio=output_audio,
    predictor_onnx=predictor_onnx,
    method=f0_method,
    hybrid_method="",
    hop_length=hop_length,
    embed_mode=embedders_mode,
    embedders=embedder_model,
    custom_embedders="",
    resample_sr=resample_sr,
    filter_radius=filter_radius,
    rms_mix_rate=rms_mix_rate,
    protect=protect,
    formant_shifting=formant_shifting,
    formant_qfrency=formant_qfrency_1,
    formant_timbre=formant_timbre_1,
    formant_qfrency_2=formant_qfrency_2,
    formant_timbre_2=formant_timbre_2,
    proposal_pitch=proposal_pitch,
    proposal_pitch_threshold=proposal_pitch_threshold,
    audio_processing=audio_processing,
    alpha=alpha,
)

print(f"\n✅ Whisper multi-speaker output saved to: {output_audio}")

<h2><font color="#38f9d7">🎵 Audio Separation (UVR)</font></h2>
<p>Separate vocals from instrumentals using UVR5 models.<br>Mirrors the <b>Separator</b> tab from the Gradio UI.</p>


In [ ]:
# @markdown ---
# @markdown ### 📂 Input & Model
# @title Audio Separation (Full Parameters)
# @markdown Separate vocals from instrumentals using UVR5 models with full parameter control.

# --- Input ---
input_audio = ""  # @param {type:"string"}
output_dir = ""  # @param {type:"string"}
uvr_model = "MDXNET_Main"  # @param ["MDXNET_Main", "MDX-NET-2", "MDX-Vocals-2", "BS-Roformer", "Roformer", "MDX-Version-1"]
output_format = "wav"  # @param ["wav", "mp3", "flac"]

# @markdown ---
# @markdown ### 🎚️ Separation Parameters
# --- Separation Options ---
sample_rate = 44100  # @param [40000, 44100, 48000]
shifts = 2  # @param {type:"slider", min:1, max:10, step:1}
overlap = 0.25  # @param {type:"slider", min:0, max:0.95, step:0.05}
segments_size = 256  # @param {type:"slider", min:32, max:1024, step:32}
aggression = 5  # @param {type:"slider", min:0, max:20, step:1}

# @markdown ---
# @markdown ### 🔧 Advanced Options
# --- Advanced Options ---
enable_tta = False  # @param {type:"boolean"}
enable_denoise = False  # @param {type:"boolean"}
separate_backing = False  # @param {type:"boolean"}
separate_reverb = False  # @param {type:"boolean"}
high_end_process = False  # @param {type:"boolean"}
enable_post_process = False  # @param {type:"boolean"}
karaoke_model = "MDX-Version-1"  # @param ["MDX-Version-1", "MDX-Version-2"]
reverb_model = "MDX-Reverb"  # @param {type:"string"}
denoise_model = "Normal"  # @param {type:"string"}
batch_size = 1  # @param {type:"slider", min:1, max:8, step:1}
hop_length_uvr = 1024  # @param {type:"slider", min:64, max:4096, step:64}
window_size = 512  # @param {type:"slider", min:64, max:2048, step:64}
post_process_threshold = 0.2  # @param {type:"slider", min:0, max:1, step:0.01}

import os
os.chdir(REPO_DIR)

from arvc.services.separate import separate_music

result = separate_music(
    input_path=input_audio,
    output_dirs=output_dir,
    export_format=output_format,
    model_name=uvr_model,
    karaoke_model=karaoke_model,
    reverb_model=reverb_model,
    denoise_model=denoise_model,
    sample_rate=int(sample_rate),
    shifts=shifts,
    batch_size=batch_size,
    overlap=str(overlap),
    aggression=aggression,
    hop_length=hop_length_uvr,
    window_size=window_size,
    segments_size=segments_size,
    post_process_threshold=post_process_threshold,
    enable_tta=enable_tta,
    enable_denoise=enable_denoise,
    high_end_process=high_end_process,
    enable_post_process=enable_post_process,
    separate_backing=separate_backing,
    separate_reverb=separate_reverb,
)

print(f"\n✅ Separation complete. Output in: {output_dir}")

<h2><font color="#fee140">🏋️ Training Pipeline</font></h2>
<p>Full training pipeline: create dataset → preprocess → extract features → create index → train.<br>Run each step in order. Mirrors the <b>Training</b> tab from the Gradio UI.</p>


In [ ]:
# @markdown ---
# @markdown ### 📦 Dataset Settings
# @title Step 1: Create Dataset
# @markdown Build a training dataset from YouTube URLs or local audio files.
source = ""  # @param {type:"string"}
output_dir = "./dataset"  # @param {type:"string"}
sample_rate = 40000  # @param ["32000", "40000", "48000"]
clean_dataset = True  # @param {type:"boolean"}
clean_strength = 0.7  # @param {type:"slider", min:0, max:1, step:0.01}
separate_vocals = True  # @param {type:"boolean"}
separator_model = "MDXNET_Main"  # @param {type:"string"}
separator_reverb = False  # @param {type:"boolean"}
reverb_model = "MDX-Reverb"  # @param {type:"string"}
skip_start = 0  # @param {type:"slider", min:0, max:300, step:1}
skip_end = 0  # @param {type:"slider", min:0, max:300, step:1}

import os
os.chdir(REPO_DIR)

# Auto-detect if source is a URL or local path
if source.startswith("http"):
    cmd = f"python -m arvc.api.cli create-dataset -u \"{source}\" -o \"{output_dir}\" --sample_rate {sample_rate}"
else:
    cmd = f"python -m arvc.api.cli create-dataset -i \"{source}\" -o \"{output_dir}\" --sample_rate {sample_rate}"

if clean_dataset:
    cmd += f" --clean_dataset --clean_strength {clean_strength}"
if not separate_vocals:
    cmd += " --no-separate"
else:
    if separator_reverb:
        cmd += " --separator_reverb"
    cmd += f" --separator_model {separator_model}"
    if separator_reverb:
        cmd += f" --reverb_model {reverb_model}"
if skip_start > 0:
    cmd += f" --skip_start {skip_start}"
if skip_end > 0:
    cmd += f" --skip_end {skip_end}"

!{cmd}

In [ ]:
# @markdown ---
# @markdown ### ✂️ Slicing Settings
# @title Step 2: Preprocess
# @markdown Slice and normalize training audio data.
# @markdown **Recommended**: Enable process_effects (high-pass filter) and use 'post' normalization for best results.
model_name = ""  # @param {type:"string"}
sample_rate = 40000  # @param ["32000", "40000", "48000"]
dataset_path = "./dataset"  # @param {type:"string"}
cpu_cores = 2  # @param {type:"slider", min:1, max:8, step:1}
cut_method = "Automatic"  # @param ["Automatic", "Simple", "Skip"]
process_effects = True  # @param {type:"boolean"}
clean_dataset = False  # @param {type:"boolean"}
clean_strength = 0.7  # @param {type:"slider", min:0, max:1, step:0.01}
chunk_len = 3.0  # @param {type:"slider", min:0.5, max:10, step:0.5}
overlap_len = 0.3  # @param {type:"slider", min:0, max:1, step:0.05}
normalization = "post"  # @param ["none", "pre", "post"]

import os
os.chdir(REPO_DIR)

cmd = f"python -m arvc.api.cli preprocess {model_name} --sample_rate {sample_rate}"
cmd += f" --dataset_path \"{dataset_path}\" --cpu_cores {cpu_cores}"
cmd += f" --cut_method {cut_method}"
if process_effects:
    cmd += " --process_effects"
if clean_dataset:
    cmd += f" --clean_dataset --clean_strength {clean_strength}"
if cut_method == "Simple":
    cmd += f" --chunk_len {chunk_len} --overlap_len {overlap_len}"
if normalization != "none":
    cmd += f" --normalization {normalization}"

!{cmd}

In [ ]:
# @markdown ---
# @markdown ### 🔬 Feature Extraction Settings
# @title Step 3: Extract Features
# @markdown Extract embeddings and F0 features from preprocessed data.
# @markdown **Recommended**: Use rmvpe for best F0 extraction quality.
model_name = ""  # @param {type:"string"}
sample_rate = 40000  # @param ["32000", "40000", "48000"]
f0_method = "rmvpe"  # @param ["rmvpe", "crepe-full", "crepe-tiny", "fcpe", "harvest", "pyin"]
rvc_version = "v2"  # @param ["v1", "v2"]
cpu_cores = 2  # @param {type:"slider", min:1, max:8, step:1}
gpu_id = "0"  # @param {type:"string"}
embedder_model = "hubert_base"  # @param {type:"string"}
embedders_mode = "fairseq"  # @param ["fairseq", "transformers", "onnx", "whisper"]
f0_onnx = False  # @param {type:"boolean"}
predictor_onnx = False  # @param {type:"boolean"}
pitch_guidance = True  # @param {type:"boolean"}
hop_length = 128  # @param {type:"slider", min:32, max:512, step:32}
rms_extract = False  # @param {type:"boolean"}

import os
os.chdir(REPO_DIR)

cmd = f"python -m arvc.api.cli extract {model_name} --sample_rate {sample_rate}"
cmd += f" --f0_method {f0_method} --version {rvc_version}"
cmd += f" --cpu_cores {cpu_cores} --gpu {gpu_id}"
cmd += f" --embedder_model {embedder_model}"
cmd += f" --hop_length {hop_length}"
if f0_onnx:
    cmd += " --f0_onnx"
if predictor_onnx:
    cmd += " --predictor_onnx"
if not pitch_guidance:
    cmd += " --no-pitch_guidance"
if rms_extract:
    cmd += " --rms_extract"

!{cmd}

In [ ]:
# @title Step 4: Create Index
# @markdown Create the .index file for voice retrieval.
model_name = ""  # @param {type:"string"}
rvc_version = "v2"  # @param ["v1", "v2"]
algorithm = "Auto"  # @param ["Auto", "Faiss", "KMeans"]

import os
os.chdir(REPO_DIR)

!python -m arvc.api.cli create-index {model_name} --version {rvc_version} --algorithm {algorithm}

In [ ]:
# @markdown ---
# @markdown ### 🏋️ Training Settings
# @title Step 5: Train Model
# @markdown Train the RVC voice model with full parameter control.
# @markdown **Important fixes applied**: gradient clipping enabled, LR warmup added, better default batch size.
# @markdown For best results on Colab T4: use batch_size=4, 200-300 epochs, rmvpe F0.
model_name = ""  # @param {type:"string"}
rvc_version = "v2"  # @param ["v1", "v2"]
epochs = 200  # @param {type:"slider", min:50, max:1000, step:50}
batch_size = 4  # @param ["2", "4", "8", "16"]
save_every = 25  # @param {type:"slider", min:5, max:100, step:5}
gpu_id = "0"  # @param {type:"string"}
author = ""  # @param {type:"string"}
optimizer = "AdamW"  # @param ["AdamW", "ScheduleFreeAdamW", "Lion", "Prodigy", "NAdam"]
vocoder = "HiFi-GAN"  # @param ["HiFi-GAN", "Default", "MRF-HiFi-GAN", "RefineGAN", "BigVGAN", "RingFormer", "PCPH-GAN", "Vocos", "HiFi-GAN-v3", "JVSF-HiFi-GAN", "WaveGlow", "NSF-APNet", "FullBand-MRF"]
overtrain_detect = False  # @param {type:"boolean"}
overtrain_threshold = 50  # @param {type:"slider", min:10, max:100, step:5}
pitch_guidance = True  # @param {type:"boolean"}
cache_gpu = True  # @param {type:"boolean"}
checkpointing = False  # @param {type:"boolean"}
multiscale_loss = False  # @param {type:"boolean"}
cosine_lr = True  # @param {type:"boolean"}
energy_use = False  # @param {type:"boolean"}
use_reference = False  # @param {type:"boolean"}
reference_path = ""  # @param {type:"string"}
pretrained_g = ""  # @param {type:"string"}
pretrained_d = ""  # @param {type:"string"}

import os
os.chdir(REPO_DIR)

cmd = f"python -m arvc.api.cli train {model_name} --version {rvc_version}"
cmd += f" --epochs {epochs} --batch_size {batch_size}"
cmd += f" --save_every {save_every} --gpu {gpu_id}"
cmd += f" --optimizer {optimizer}"
if vocoder != "HiFi-GAN":
    cmd += f" --vocoder {vocoder}"
if author:
    cmd += f" --author \"{author}\""
if overtrain_detect:
    cmd += f" --overtrain_detect --overtrain_threshold {overtrain_threshold}"
if not pitch_guidance:
    cmd += " --no-pitch_guidance"
if cache_gpu:
    cmd += " --cache_gpu"
if checkpointing:
    cmd += " --checkpointing"
if multiscale_loss:
    cmd += " --multiscale_loss"
if cosine_lr:
    cmd += " --cosine_lr"
if energy_use:
    cmd += " --energy"
if use_reference and reference_path:
    cmd += f" --use_reference --reference_path \"{reference_path}\""
if pretrained_g:
    cmd += f" --pretrained_g \"{pretrained_g}\""
if pretrained_d:
    cmd += f" --pretrained_d \"{pretrained_d}\""

!{cmd}

In [ ]:
# @markdown ---
# @markdown ### 🚀 One-Click Pipeline Settings
# @title One-Click Training (Full Pipeline)
# @markdown Run the entire training pipeline in one step: preprocess → extract → train → create index.
# @markdown **Uses optimized defaults** for Colab: batch_size=4, rmvpe F0, post-normalization, gradient clipping, LR warmup.
model_name = ""  # @param {type:"string"}
rvc_version = "v2"  # @param ["v1", "v2"]
sample_rate = "40k"  # @param ["32k", "40k", "48k"]
dataset_path = "./dataset"  # @param {type:"string"}
pitch_guidance = True  # @param {type:"boolean"}
f0_method = "rmvpe"  # @param ["rmvpe", "crepe-full", "crepe-tiny", "fcpe", "harvest", "pyin"]
total_epoch = 200  # @param {type:"slider", min:50, max:1000, step:50}
batch_size = 4  # @param ["2", "4", "8"]
save_every = 25  # @param {type:"slider", min:5, max:100, step:5}
gpu = "0"  # @param {type:"string"}
vocoder = "HiFi-GAN"  # @param ["HiFi-GAN", "Default", "MRF-HiFi-GAN", "RefineGAN", "BigVGAN"]
optimizer = "AdamW"  # @param ["AdamW", "ScheduleFreeAdamW", "Lion", "Prodigy", "NAdam"]
model_author = ""  # @param {type:"string"}

import os, sys
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from arvc.services.training import one_click_train

print(f"Starting one-click training for '{model_name}'...")
print(f"  Version: {rvc_version}, SR: {sample_rate}, Epochs: {total_epoch}")
print(f"  F0: {f0_method}, Batch: {batch_size}, Optimizer: {optimizer}")
print()

for log in one_click_train(
    model_name=model_name,
    rvc_version=rvc_version,
    sample_rate=sample_rate,
    dataset_path=dataset_path,
    pitch_guidance=pitch_guidance,
    f0_method=f0_method,
    total_epoch=total_epoch,
    batch_size=int(batch_size),
    save_every=save_every,
    gpu=gpu,
    vocoder=vocoder,
    optimizer=optimizer,
    model_author=model_author,
):
    clear_output(wait=True)
    print(log)

print(f"\n✅ One-click training complete for '{model_name}'!")

In [ ]:
# @title Create Reference Set
# @markdown Create a reference audio set for improved inference quality.
audio_file = ""  # @param {type:"string"}
ref_name = "reference"  # @param {type:"string"}
f0_method = "rmvpe"  # @param ["rmvpe", "crepe-full", "fcpe", "harvest"]
pitch_shift = 0  # @param {type:"slider", min:-24, max:24, step:1}
rvc_version = "v2"  # @param ["v1", "v2"]
embedder_model = "hubert_base"  # @param {type:"string"}
f0_autotune = False  # @param {type:"boolean"}
alpha = 0.5  # @param {type:"slider", min:0, max:1, step:0.01}

import os
os.chdir(REPO_DIR)

cmd = f"python -m arvc.api.cli create-ref \"{audio_file}\" -n {ref_name}"
cmd += f" --f0_method {f0_method} --version {rvc_version}"
cmd += f" --embedder_model {embedder_model}"
if pitch_shift != 0:
    cmd += f" --pitch_shift {pitch_shift}"
if f0_autotune:
    cmd += " --f0_autotune"
if alpha != 0.5:
    cmd += f" --alpha {alpha}"

!{cmd}

<h2><font color="#fbc2eb">🔧 Extra Tools</font></h2>
<p>Model fusion, F0 extraction, ONNX conversion, model info, SRT creation, and process management.<br>Mirrors the <b>Extra</b> tab from the Gradio UI.</p>


In [ ]:
# @title Model Fusion
# @markdown Merge two RVC models together with a blending ratio. Mirrors the Fusion tab.
model_path_a = ""  # @param {type:"string"}
model_path_b = ""  # @param {type:"string"}
output_name = "fused_model"  # @param {type:"string"}
ratio = 0.5  # @param {type:"slider", min:0, max:1, step:0.01}

import os
os.chdir(REPO_DIR)

from arvc.services.model_utils import fushion_model

result = fushion_model(
    name_to_save=output_name,
    model_path_a=model_path_a,
    model_path_b=model_path_b,
    ratio=ratio,
)
print(f"Fusion result: {result}")

In [ ]:
# @title F0 Extract (Standalone)
# @markdown Extract F0 (pitch) data from an audio file. Mirrors the F0 Extractor tab.
input_audio = ""  # @param {type:"string"}
f0_method = "rmvpe"  # @param ["rmvpe", "crepe-full", "crepe-tiny", "fcpe", "harvest", "pyin", "mangio-crepe-full"]
onnx_f0_mode = False  # @param {type:"boolean"}

import os
os.chdir(REPO_DIR)

from arvc.services.f0_extract import f0_extract

result = f0_extract(
    input_audio_path=input_audio,
    f0_method_extract=f0_method,
    onnx_f0_mode=onnx_f0_mode,
)
print(f"F0 extraction result: {result}")

In [ ]:
# @title Convert Model to ONNX
# @markdown Export a PyTorch (.pth) RVC model to ONNX format. Mirrors the Convert Model tab.
model_pth_path = ""  # @param {type:"string"}

import os
os.chdir(REPO_DIR)

from arvc.services.model_utils import onnx_export

result = onnx_export(model_pth_path=model_pth_path)
print(f"ONNX export result: {result}")

In [ ]:
# @title Read Model Info
# @markdown Inspect an RVC model's metadata (version, sample rate, embedder, etc.). Mirrors the Read Model tab.
model_path = ""  # @param {type:"string"}

import os
os.chdir(REPO_DIR)

from arvc.services.model_utils import model_info

info = model_info(model_path=model_path)
print(info)

In [ ]:
# @title Create SRT Subtitles
# @markdown Generate SRT subtitle files from audio using Whisper. Mirrors the Create SRT tab.
input_audio = ""  # @param {type:"string"}
output_file = "srt/output.srt"  # @param {type:"string"}
model_size = "medium"  # @param ["tiny", "tiny.en", "base", "base.en", "small", "small.en", "medium", "medium.en", "large-v1", "large-v2", "large-v3", "large-v3-turbo"]
word_timestamps = False  # @param {type:"boolean"}

import os
os.chdir(REPO_DIR)

from arvc.services.csrt import create_srt

result = create_srt(
    model_size=model_size,
    input_audio=input_audio,
    output_file=output_file,
    word_timestamps=word_timestamps,
)
print(f"SRT creation result: {result}")

<h2><font color="#2575fc">⚙️ Settings & Process Control</font></h2>
<p>Change precision, stop running processes, and manage settings.<br>Mirrors the <b>Settings</b> tab from the Gradio UI.</p>


In [ ]:
# @title Set Precision (fp16 / fp32)
# @markdown Change floating-point precision. fp16 uses less VRAM, fp32 is more accurate.
precision = "fp16"  # @param ["fp16", "fp32"]

import os, json
os.chdir(REPO_DIR)

config_path = "arvc/assets/config.txt"
if os.path.exists(config_path):
    with open(config_path, "r") as f:
        config = json.load(f)
else:
    config = {}

config["fp16"] = (precision == "fp16")
with open(config_path, "w") as f:
    json.dump(config, f, indent=2)

print(f"✅ Precision set to {precision}")
print("Note: Restart any running processes for this to take effect.")

In [ ]:
# @title Stop Running Processes
# @markdown Kill running separation, conversion, dataset creation, or training processes.
stop_separate = False  # @param {type:"boolean"}
stop_convert = False  # @param {type:"boolean"}
stop_create_dataset = False  # @param {type:"boolean"}
stop_preprocess = False  # @param {type:"boolean"}
stop_extract = False  # @param {type:"boolean"}
stop_train = False  # @param {type:"boolean"}
model_name_for_stop = ""  # @param {type:"string"}

import os
os.chdir(REPO_DIR)

from arvc.services.utils import stop_pid

if stop_separate:
    stop_pid("separate_pid", None, False)
    print("✅ Separation process stopped.")
if stop_convert:
    stop_pid("convert_pid", None, False)
    print("✅ Conversion process stopped.")
if stop_create_dataset:
    stop_pid("create_dataset_pid", None, False)
    print("✅ Dataset creation process stopped.")
if stop_preprocess:
    stop_pid("preprocess_pid", model_name_for_stop, False)
    print("✅ Preprocess process stopped.")
if stop_extract:
    stop_pid("extract_pid", model_name_for_stop, False)
    print("✅ Extract process stopped.")
if stop_train:
    stop_pid("train_pid", model_name_for_stop, True)
    print("✅ Training process stopped.")

if not any([stop_separate, stop_convert, stop_create_dataset, stop_preprocess, stop_extract, stop_train]):
    print("No processes selected to stop.")

<h2><font color="#3cba92">💾 Backup & Restore</font></h2>
<p>Backup models to Google Drive and restore them later.</p>


In [ ]:
# @title Backup Model to Drive
# @markdown Copy a trained model from Colab to your Google Drive for safekeeping.
model_name = "" # @param {type:"string"}

import os, shutil
from google.colab import drive
from google.colab._message import MessageError

try:
  drive.mount("/content/drive")
except MessageError:
  pass

src = f"{LOGS_PATH}/{model_name}"
dst = f"{BACKUPS_PATH}/{model_name}"

if not model_name:
    print("❌ Enter a model name.")
elif not os.path.exists(src):
    print(f"❌ Model '{model_name}' not found at {src}")
else:
    os.makedirs(BACKUPS_PATH, exist_ok=True)
    if os.path.exists(dst):
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    print(f"✅ Backed up '{model_name}' to Google Drive.")

In [ ]:
# @title List Backed Up Models

import os

if not os.path.exists(BACKUPS_PATH):
    print(f"❌ Backup directory not found: {BACKUPS_PATH}")
else:
    models = [d for d in os.listdir(BACKUPS_PATH) if os.path.isdir(os.path.join(BACKUPS_PATH, d))]
    if not models:
        print("No backed up models found.")
    else:
        for i, m in enumerate(models):
            size = sum(os.path.getsize(os.path.join(BACKUPS_PATH, m, f)) for f in os.listdir(os.path.join(BACKUPS_PATH, m)) if os.path.isfile(os.path.join(BACKUPS_PATH, m, f)))
            size_mb = size / (1024*1024)
            print(f"  {i+1}. {m} ({size_mb:.1f} MB)")

In [ ]:
# @title Restore Model from Drive
# @markdown Load a previously backed up model from Google Drive into Colab.
model_name = "" # @param {type:"string"}

import os, shutil

src = f"{BACKUPS_PATH}/{model_name}"
dst = f"{LOGS_PATH}/{model_name}"

if not model_name:
    print("❌ Enter a model name.")
elif not os.path.exists(src):
    print(f"❌ Backup '{model_name}' not found in Google Drive.")
else:
    os.makedirs(LOGS_PATH, exist_ok=True)
    if os.path.exists(dst):
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    print(f"✅ Restored '{model_name}' from Google Drive.")

<h2><font color="#ee0979">📤 Push to HuggingFace</font></h2>
<p>Upload your trained model to HuggingFace to share it with others.</p>


In [ ]:
# @title Push Model to HuggingFace
# @markdown Upload your trained model. Get a token from https://huggingface.co/settings/tokens
hf_token = "" #@param {type:"string"}
repo_id = "" #@param {type:"string"}
model_name = "" #@param {type:"string"}

import os, shutil
from huggingface_hub import HfApi, create_repo, login

login(hf_token)
api = HfApi()

try:
    create_repo(repo_id)
except Exception as e:
    print(f"Note: {e}. Proceeding...")

model_path = f"{LOGS_PATH}/{model_name}"
zip_path = f"{model_path}/{model_name}.zip"

if not os.path.exists(zip_path):
    print(f"Zipping {model_path}...")
    shutil.make_archive(f"{model_path}/{model_name}", 'zip', model_path)
    zip_path = f"{model_path}/{model_name}.zip"

api.upload_file(
    path_or_fileobj=zip_path,
    path_in_repo=f"{model_name}.zip",
    repo_id=repo_id,
    repo_type="model"
)

print(f"\n✅ Model uploaded to https://huggingface.co/{repo_id}")

<h2><font color="#9bc5c3">📖 CLI Quick Reference</font></h2>
